# 기본 53열 데이터와 확장 126열 데이터 비교

동일 학생 단위 3-Fold, 동일 Target, 동일 하이퍼파라미터에서 기본 Feature Set(모델 입력 51개)과 확장 Feature Set(모델 입력 124개)을 비교한다.

- 기본 데이터: `../data/oulad_weekly_next_week_base.csv`
- 확장 데이터: `../data/oulad_weekly_next_week.csv`
- 비교 모델: XGBoost, CatBoost
- 평가지표: PR-AUC, Recall@Top-20%, Precision@Top-20%, Brier Score, ECE
- 결과 저장: 현재 `feature_set_comparison/` 폴더

> 실행 시간이 길면 `04_compare_base_vs_enhanced_features.py`의 `MODELS_TO_RUN`을 `['xgboost']`로 바꿔 1차 확인한 뒤 CatBoost를 추가 실행한다.

In [ ]:
import runpy
from pathlib import Path

# 프로젝트 루트 또는 notebooks 폴더에서 실행해도 비교 스크립트를 찾도록 처리한다.
script_path = Path.cwd() / '04_compare_base_vs_enhanced_features.py'
if not script_path.exists():
    script_path = Path.cwd() / 'models' / 'feature_set_comparison' / '04_compare_base_vs_enhanced_features.py'
runpy.run_path(str(script_path), run_name='__main__')

## 선택 기준

1. CatBoost 결과를 우선으로 보고, 확장 데이터가 PR-AUC와 Recall@Top-20%를 일관되게 개선하는지 확인한다.
2. Precision@Top-20%와 Brier·ECE가 크게 나빠지면 개입 인원과 확률 신뢰도를 함께 고려한다.
3. Fold별 지표가 한 Fold에서만 좋아진 결과인지 확인한다.
4. 확장 데이터를 채택한 뒤에만 RandomForest를 추가 비교하고, 최종 CatBoost에는 별도 Validation Set 기반 확률 보정을 적용한다.

In [ ]:
import pandas as pd

# 비교 결과 CSV는 현재 노트북과 같은 feature_set_comparison 폴더에 저장된다.
pd.read_csv('base_vs_enhanced_metrics.csv', encoding='utf-8-sig')

## 결과
확장 데이터(124 feature)는 두 모델 모두에서:
- PR-AUC 상승
- 상위 위험군 20% 안에서 이탈자 포착률 상승
- 위험군 Precision 상승
- XGBoost의 Brier·ECE도 개선
#### 따라서 확장 데이터(124 feature) 선택